# CoinGlass API Connectivity Check

Goals:
1. Verify API connectivity (successful request)
2. Print response headers related to per-minute request limits
3. Fetch Crypto Fear & Greed Index as an example

In [ ]:
import os
import requests
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

CONFIG = {
    'api_key': os.getenv('COINGLASS_API_KEY'),
    'base_url': 'https://open-api-v4.coinglass.com',
}

if not CONFIG['api_key']:
    raise ValueError('Please set the COINGLASS_API_KEY environment variable first.')

headers = {
    'CG-API-KEY': CONFIG['api_key'],
    'accept': 'application/json',
}

endpoint = '/api/index/fear-greed-history'

resp = requests.get(f"{CONFIG['base_url']}{endpoint}", headers=headers, timeout=30)
print('HTTP status code:', resp.status_code)
print('Request URL:', resp.url)

resp.raise_for_status()
payload = resp.json()
print('Business status code:', payload.get('code'))
print('Business message:', payload.get('msg'))

if str(payload.get('code')) != '0':
    raise RuntimeError(f"Request failed: code={payload.get('code')} msg={payload.get('msg')}")

In [ ]:
import matplotlib.pyplot as plt

# Print rate-limit-related response headers (case-insensitive)
rate_headers = {
    k: v for k, v in resp.headers.items()
    if 'limit' in k.lower() or 'rate' in k.lower() or 'api-key' in k.lower()
}

print('\n=== Rate-limit-related response headers ===')
if rate_headers:
    for k, v in rate_headers.items():
        print(f'{k}: {v}')
else:
    print('No limit-related headers were found. Please check your subscription plan or API docs.')

data = payload.get('data', {})
if not isinstance(data, dict):
    raise ValueError('Unexpected payload format: data is not a dictionary.')

fg_values = data.get('data_list', [])
price_values = data.get('price_list', [])
time_values = data.get('time_list', [])

if not (isinstance(fg_values, list) and isinstance(price_values, list) and isinstance(time_values, list)):
    raise ValueError('Unexpected payload format: expected list fields data_list/price_list/time_list.')

n = min(len(fg_values), len(price_values), len(time_values))
if n == 0:
    raise ValueError('No Fear & Greed data returned from API.')

df = pd.DataFrame({
    'time_ms': pd.to_numeric(time_values[:n], errors='coerce'),
    'fear_greed_index': pd.to_numeric(fg_values[:n], errors='coerce'),
    'btc_price': pd.to_numeric(price_values[:n], errors='coerce'),
}).dropna()

df['time_utc'] = pd.to_datetime(df['time_ms'], unit='ms', utc=True)
df = df.sort_values('time_utc').reset_index(drop=True)

print(f"\nFetched rows: {len(df)}")
display(df.tail(5))

fig, ax1 = plt.subplots(figsize=(12, 5))

# Area chart with 50 as threshold:
# - green area for values above 50
# - red area for values below 50
baseline = 50
x = df['time_utc']
y = df['fear_greed_index']

ax1.fill_between(x, baseline, y, where=(y >= baseline), color='green', alpha=0.35, interpolate=True, label='Index >= 50')
ax1.fill_between(x, baseline, y, where=(y < baseline), color='red', alpha=0.35, interpolate=True, label='Index < 50')
ax1.axhline(baseline, color='gray', linestyle='--', linewidth=1, alpha=0.9, label='Threshold = 50')

ax1.set_xlabel('Date (UTC)')
ax1.set_ylabel('Fear & Greed Index')
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(df['time_utc'], df['btc_price'], color='tab:orange', linewidth=1.4, label='BTC Price')
ax2.set_ylabel('BTC Price (USD)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

ax1.set_title('Crypto Fear & Greed Index (Area, Threshold=50) vs BTC Price')

# Combine legends from both axes
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left')

plt.tight_layout()
plt.show()